# OHLCV Download – Paper Tickers

Downloads adjusted OHLCV for the 5 stocks used in Chen & Kawashima (2025):  
`AAPL, HSBC, PEP, TM, TCEHY` | **2016-05-01 to 2024-05-08**   
Source: **yfinance** (Yahoo Finance)  
Output: `data/price/<TICKER>.csv`

In [1]:
import os
import warnings
import pandas as pd
import yfinance as yf
warnings.filterwarnings('ignore')

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
OUTPUT_DIR = os.path.join(ROOT, 'data', 'price')
os.makedirs(OUTPUT_DIR, exist_ok=True)

PAPER_TICKERS = ['AAPL', 'HSBC', 'PEP', 'TM', 'TCEHY']
START = '2016-05-01'
END   = '2024-05-09'   # exclusive in yfinance

print('Output dir:', OUTPUT_DIR)
print('Tickers   :', PAPER_TICKERS)
print('Range     :', START, '->', END)

Output dir: /Users/tmq/Documents/GitHub/stocks-prediction/data/price
Tickers   : ['AAPL', 'HSBC', 'PEP', 'TM', 'TCEHY']
Range     : 2016-05-01 -> 2024-05-09


In [2]:
summary = []

for ticker in PAPER_TICKERS:
    print(f'[{ticker}] downloading ...', end=' ', flush=True)
    try:
        raw = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
    except Exception as e:
        print(f'ERROR: {e}')
        continue

    if raw.empty:
        print('WARNING: empty response')
        continue

    # Flatten MultiIndex columns if present
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [col[0].lower() for col in raw.columns]
    else:
        raw.columns = [c.lower() for c in raw.columns]

    raw = raw.reset_index()
    if 'Date' in raw.columns:
        raw = raw.rename(columns={'Date': 'date'})
    raw['date'] = pd.to_datetime(raw['date']).dt.strftime('%Y-%m-%d')
    raw['ticker'] = ticker

    cols = ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']
    raw = raw[[c for c in cols if c in raw.columns]]
    raw.to_csv(os.path.join(OUTPUT_DIR, f'{ticker}.csv'), index=False)

    summary.append({'ticker': ticker, 'rows': len(raw),
                    'start': raw['date'].min(), 'end': raw['date'].max()})
    print(f'OK - {len(raw)} rows ({raw["date"].min()} -> {raw["date"].max()})')

pd.DataFrame(summary)

[AAPL] downloading ... OK - 2019 rows (2016-05-02 -> 2024-05-08)
[HSBC] downloading ... OK - 2019 rows (2016-05-02 -> 2024-05-08)
[PEP] downloading ... OK - 2019 rows (2016-05-02 -> 2024-05-08)
[TM] downloading ... OK - 2019 rows (2016-05-02 -> 2024-05-08)
[TCEHY] downloading ... OK - 2019 rows (2016-05-02 -> 2024-05-08)


,ticker,rows,start,end
0,AAPL,2019,2016-05-02,2024-05-08
1,HSBC,2019,2016-05-02,2024-05-08
2,PEP,2019,2016-05-02,2024-05-08
3,TM,2019,2016-05-02,2024-05-08
4,TCEHY,2019,2016-05-02,2024-05-08


## Data quality check

In [3]:
# Per-ticker: shape, date range, missing values, zero-volume days
rows = []
for ticker in PAPER_TICKERS:
    path = os.path.join(OUTPUT_DIR, f'{ticker}.csv')
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date'])
    nan_count  = df[['open','high','low','close','volume']].isna().sum().sum()
    zero_vol   = (df['volume'] == 0).sum()
    rows.append({
        'ticker': ticker,
        'rows': len(df),
        'start': df['date'].min().date(),
        'end': df['date'].max().date(),
        'nan_cells': nan_count,
        'zero_vol_days': zero_vol,
    })
    print(f'[{ticker}] {len(df)} rows | NaN={nan_count} | zero_vol={zero_vol}')

pd.DataFrame(rows)

[AAPL] 2019 rows | NaN=0 | zero_vol=0
[HSBC] 2019 rows | NaN=0 | zero_vol=0
[PEP] 2019 rows | NaN=0 | zero_vol=0
[TM] 2019 rows | NaN=0 | zero_vol=0
[TCEHY] 2019 rows | NaN=0 | zero_vol=0


,ticker,rows,start,end,nan_cells,zero_vol_days
0,AAPL,2019,2016-05-02,2024-05-08,0,0
1,HSBC,2019,2016-05-02,2024-05-08,0,0
2,PEP,2019,2016-05-02,2024-05-08,0,0
3,TM,2019,2016-05-02,2024-05-08,0,0
4,TCEHY,2019,2016-05-02,2024-05-08,0,0


In [4]:
# Check for large date gaps (> 5 calendar days = potential missing trading days)
for ticker in PAPER_TICKERS:
    df = pd.read_csv(os.path.join(OUTPUT_DIR, f'{ticker}.csv'), parse_dates=['date'])
    df = df.sort_values('date').reset_index(drop=True)
    gaps = df['date'].diff().dt.days
    big_gaps = gaps[gaps > 5].reset_index()
    if len(big_gaps) == 0:
        print(f'[{ticker}] no suspicious gaps')
    else:
        for _, row in big_gaps.iterrows():
            d = df.loc[int(row['index']), 'date'].date()
            print(f'[{ticker}] gap of {int(row["date"])} days before {d}')

[AAPL] no suspicious gaps
[HSBC] no suspicious gaps
[PEP] no suspicious gaps
[TM] no suspicious gaps
[TCEHY] no suspicious gaps


In [5]:
# Sanity check: TCEHY is OTC (low price), others are USD-listed
# Check for obviously wrong prices (< $0.01 or > $10,000)
for ticker in PAPER_TICKERS:
    df = pd.read_csv(os.path.join(OUTPUT_DIR, f'{ticker}.csv'))
    bad = df[(df['close'] < 0.01) | (df['close'] > 10000)]
    if len(bad) == 0:
        close_range = f'{df["close"].min():.2f} - {df["close"].max():.2f}'
        print(f'[{ticker}] close range OK: ${close_range}')
    else:
        print(f'[{ticker}] WARNING: {len(bad)} rows with suspicious close price')
        print(bad.head())

[AAPL] close range OK: $20.58 - 196.07
[HSBC] close range OK: $13.05 - 39.51
[PEP] close range OK: $73.92 - 177.25
[TM] close range OK: $76.55 - 243.76
[TCEHY] close range OK: $17.37 - 89.03
